In [1]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_current_matchup, get_roster_by_team, get_league_categories
from fantasy.nba_stats import get_player_stats, get_games_this_week, batch_get_player_stats
from fantasy.analysis import project_team_categories, classify_categories
from fantasy.llm import build_matchup_prompt, ask_gemini
import pandas as pd

In [2]:
matchup = get_current_matchup()
my_roster = get_my_roster()
opp_roster = get_roster_by_team(matchup["opponent_team_key"])
categories = get_league_categories()
week_start, week_end = matchup["week_start"], matchup["week_end"]
print(f"Week {matchup['week']}: {week_start} → {week_end}")
print(f"Opponent team: {matchup['opponent_team_key']}")

[2026-03-23 22:47:20,718 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-23 22:47:20,722 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 790.1353933811188
[2026-03-23 22:47:20,723 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-23 22:47:20,736 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 790.1496689319611
[2026-03-23 22:47:20,737 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-23 22:47:21,174 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-23 22:47:21,177 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 790.5907554626465
[2026-03-23 22:47:21,178 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-23 22:47:21,185 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 790.598557472229
[2026-03-23 22:47:21,186 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID
[2026-03-23 22:47:21,868 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-23 22:47:21,872 DEBUG] [yah

Week 22: 2026-03-23 → 2026-03-29
Opponent team: 466.l.26708.t.5


In [3]:
def enrich_players(roster):
    # Fetch stats in parallel, then get game counts
    with_stats = batch_get_player_stats(roster)
    return [{**p, "games_remaining": get_games_this_week(p["team_abbr"], week_start, week_end)}
            for p in with_stats]

my_players = enrich_players(my_roster)
opp_players = enrich_players(opp_roster)

In [4]:
my_proj = project_team_categories(my_players)
opp_proj = project_team_categories(opp_players)

def to_league_cats(proj, cats):
    """Map raw stat projections to league scoring category values."""
    result = {}
    for cat in cats:
        if cat in proj:
            result[cat] = proj[cat]
        elif cat == "FG%":
            result[cat] = proj["FGM"] / proj["FGA"] if proj.get("FGA", 0) > 0 else 0.0
        elif cat == "FT%":
            result[cat] = proj["FTM"] / proj["FTA"] if proj.get("FTA", 0) > 0 else 0.0
        elif cat == "3PTM":
            result[cat] = proj.get("FG3M", 0.0)
    return result

my_cats = to_league_cats(my_proj, categories)
opp_cats = to_league_cats(opp_proj, categories)
classification = classify_categories(my_cats, opp_cats)

df = pd.DataFrame({
    "My Team": my_cats,
    "Opponent": opp_cats,
    "Status": classification,
}).round(3)
print(df.to_string())

      My Team  Opponent  Status
FG%       0.0       0.0  tossup
FT%       0.0       0.0  tossup
3PTM      0.0       0.0  tossup
PTS       0.0       0.0  tossup
REB       0.0       0.0  tossup
AST       0.0       0.0  tossup
BLK       0.0       0.0  tossup


In [5]:
prompt = build_matchup_prompt(my_cats, opp_cats, classification)
advice = ask_gemini(prompt)
print("\n=== WEEKLY GAME PLAN ===\n")
print(advice)


=== WEEKLY GAME PLAN ===

Okay, based on the current "tossup" projection across the board, we need a defined strategy to gain an edge. We can't realistically target every category, so let's focus on leveraging specific strengths or weaknesses in our existing roster.

**Overall Strategy:**

Given the toss-up status, I recommend a **"Rebound & Block" Strategy** This means we'll aggressively pursue REB and BLK while solidifying our chances in AST. Other categories will be opportunistic pick-ups.

*   **Categories to Win:** REB, BLK, AST
*   **Categories to Punt:** FG% (if necessary, will be opportunistic)

**Rationale:**

REB and BLK are often undervalued categories. If we can dominate them, we create a significant weekly advantage. AST, while a toss-up, will be easier to secure with active guard streaming and lineup management.

**Specific Actions & Streaming Targets:**

1.  **Rebound & Block Streaming:**
    *   Prioritize players who provide high rebounding and block rates in limited 